# Embedding Self-Similarity — Simplex vs. Root+*hata* Minimal Pairs

Mean pairwise cosine similarity (Ethayarajh 2019) across usage sentences, for three embedding models and two registers (dictionary examples, NSMC).

In [1]:
import pandas as pd

dict_ex = pd.read_csv("self_similarity_dict_examples_results.csv")
dict_ex["register"] = "dict"
nsmc = pd.read_csv("self_similarity_nsmc_results.csv")
nsmc["register"] = "nsmc"
vertex = pd.read_csv("self_similarity_vertex_results.csv")
vertex["register"] = "dict"

all_results = pd.concat([dict_ex, nsmc, vertex], ignore_index=True)
model_short = {"klue/bert-base": "klue", "beomi/kcbert-base": "kcbert",
               "text-multilingual-embedding-002": "vertex"}
all_results["condition"] = all_results["model"].map(model_short) + "-" + all_results["register"]
all_results.head()

,model,word,n_examples,self_similarity,register,condition
0,klue/bert-base,헤아리다,26,0.415312,dict,klue-dict
1,klue/bert-base,생각하다,45,0.590861,dict,klue-dict
2,klue/bert-base,부르다,49,0.482811,dict,klue-dict
3,klue/bert-base,노래하다,20,0.598859,dict,klue-dict
4,klue/bert-base,게우다,12,0.604352,dict,klue-dict


In [2]:
wide = all_results.pivot_table(index="word", columns="condition", values="self_similarity", aggfunc="first")
col_order = [c for c in ["klue-dict", "kcbert-dict", "vertex-dict", "klue-nsmc", "kcbert-nsmc"] if c in wide.columns]
wide[col_order].round(4)

condition,klue-dict,kcbert-dict,vertex-dict,klue-nsmc,kcbert-nsmc
word,,,,,
게우다,0.6044,0.5377,0.7219,NaN,NaN
굽다,0.4041,0.3919,0.6643,0.3511,0.3118
노래하다,0.5989,0.5691,0.7150,0.6409,0.4984
덥다,0.5449,0.5778,0.7228,0.5246,0.5292
따뜻하다,0.6640,0.5091,0.7081,0.6684,0.5754
부르다,0.4828,0.4186,0.6864,0.3441,0.3600
생각하다,0.5909,0.4325,0.6713,0.5333,0.3330
요리하다,0.5391,0.5413,0.7493,0.6312,0.4151
토하다,0.5237,0.4920,0.7179,0.5787,0.5322


In [3]:
PAIRS = [("헤아리다", "생각하다", "think"), ("부르다", "노래하다", "sing"),
         ("게우다", "토하다", "vomit"), ("굽다", "요리하다", "cook"),
         ("덥다", "따뜻하다", "warm")]

rows = []
for simplex, hada, concept in PAIRS:
    row = {"concept": concept, "simplex": simplex, "hada": hada}
    n_match, n_total = 0, 0
    for cond in col_order:
        s, h = wide.loc[simplex, cond], wide.loc[hada, cond]
        if pd.isna(s) or pd.isna(h):
            row[cond] = "n/a"
            continue
        n_total += 1
        if s < h:
            row[cond] = "match"
            n_match += 1
        else:
            row[cond] = "opposite"
    row["match_ratio"] = f"{n_match}/{n_total}"
    rows.append(row)

pd.DataFrame(rows).set_index("concept")

,simplex,hada,klue-dict,kcbert-dict,vertex-dict,klue-nsmc,kcbert-nsmc,match_ratio
concept,,,,,,,,
think,헤아리다,생각하다,match,opposite,match,opposite,opposite,2/5
sing,부르다,노래하다,match,match,match,match,match,5/5
vomit,게우다,토하다,opposite,opposite,opposite,n/a,n/a,0/3
cook,굽다,요리하다,match,match,match,match,match,5/5
warm,덥다,따뜻하다,match,opposite,opposite,match,match,3/5
